In [3]:
import pandas as pd
import numpy as np

clientes = pd.read_parquet("data/clean/clientes_modelo.parquet")

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
estado   = [f"estado_pago_{m}" for m in meses]
facturas = [f"importe_factura_{m}" for m in meses]
pagos    = [f"importe_pagado_{m}" for m in meses]

variables = pd.DataFrame({"id_cliente": clientes["id_cliente"]})

In [4]:
variables["retraso_sep"] = clientes["estado_pago_sep"].clip(lower=-1, upper=2)
variables["meses_con_retraso"] = (clientes[estado] > 0).sum(axis=1)
variables["peor_retraso_6m"] = clientes[estado].max(axis=1).clip(lower=0)

In [5]:
limite = clientes["limite_credito"].fillna(clientes["limite_credito"].median())
variables["limite_desconocido"] = clientes["limite_credito"].isna().astype(int)
variables["limite_credito"] = limite

# facturas vacías: se rellenan con la media de ese mismo cliente
fact = clientes[facturas]
fact = fact.T.fillna(fact.mean(axis=1)).T.fillna(0)

variables["uso_limite_sep"] = (fact["importe_factura_sep"] / limite).clip(-1, 2)
variables["uso_medio_6m"] = fact.div(limite, axis=0).mean(axis=1).clip(-1, 2)

total_facturado = fact.clip(lower=0).sum(axis=1)
total_pagado = clientes[pagos].sum(axis=1)
variables["parte_pagada_6m"] = np.where(
    total_facturado > 0, (total_pagado / total_facturado).clip(upper=1), 1
)

variables["cambio_uso"] = (
    (fact["importe_factura_sep"] - fact["importe_factura_abr"]) / limite
).clip(-2, 2)

print("Huecos que quedan:", variables.isna().sum().sum())

Huecos que quedan: 0


In [6]:
edad = clientes["edad"]
variables["edad_desconocida"] = edad.isna().astype(int)
variables["edad"] = edad.fillna(edad.median())

categorias = pd.get_dummies(
    clientes[["nivel_estudios", "estado_civil"]], drop_first=True, dtype=int
)
variables = pd.concat([variables, categorias.set_index(variables.index)], axis=1)

variables["sexo"] = clientes["sexo"]
variables["impago_mes_siguiente"] = clientes["impago_mes_siguiente"]
variables.head()

,id_cliente,retraso_sep,meses_con_retraso,peor_retraso_6m,limite_desconocido,limite_credito,uso_limite_sep,uso_medio_6m,parte_pagada_6m,cambio_uso,...,edad,nivel_estudios_desconocido,nivel_estudios_otros,nivel_estudios_postgrado,nivel_estudios_universidad,estado_civil_desconocido,estado_civil_otros,estado_civil_soltero,sexo,impago_mes_siguiente
0,1,2.0,2,2.0,0,20000.0,0.195650,0.087688,0.000000,0.195650,...,24.0,0,0,0,1,0,0,0,mujer,1.0
1,2,-1.0,2,2.0,0,120000.0,0.022350,0.023718,0.292791,-0.004825,...,26.0,0,0,0,1,0,0,1,mujer,1.0
2,3,0.0,0,0.0,0,90000.0,0.324878,0.188246,0.108388,0.152111,...,34.0,0,0,0,1,0,0,1,mujer,0.0
3,4,0.0,0,0.0,0,50000.0,0.939800,0.771113,0.036259,0.348860,...,37.0,0,0,0,1,0,0,0,mujer,0.0
4,5,-1.0,0,0.0,0,50000.0,0.172340,0.364463,0.540054,-0.210280,...,57.0,0,0,0,1,0,0,0,hombre,0.0


In [7]:
numericas = variables.drop(columns=["id_cliente", "sexo", "impago_mes_siguiente"])
relacion = numericas.corrwith(variables["impago_mes_siguiente"]).sort_values(key=abs, ascending=False)
display(relacion.round(3).to_frame("relación con el impago"))

variables.to_parquet("data/clean/variables_modelo.parquet", index=False)
print("Guardado:", variables.shape)

,relación con el impago
meses_con_retraso,0.398
peor_retraso_6m,0.371
retraso_sep,0.351
limite_credito,-0.153
uso_medio_6m,0.115
parte_pagada_6m,-0.090
uso_limite_sep,0.087
nivel_estudios_postgrado,-0.052
nivel_estudios_desconocido,-0.038
nivel_estudios_universidad,0.037


Guardado: (29909, 21)
